# 🏃 Ćwiczenie 1: EDA — Statystyki sportowe (maratony amatorskie)

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 1 (Studia stacjonarne)
**Szacowany czas:** ⏱️ ~90 minut
**Poziom trudności:** ⭐ (Podstawowy)

## 📚 Cele ćwiczenia

Po wykonaniu tego notebooka będziesz potrafić:

1. Wygenerować i wczytać syntetyczny zbiór danych sportowych
2. Obliczyć miary tendencji centralnej i rozproszenia
3. Wykryć wartości odstające metodą IQR
4. Stworzyć histogramy, boxploty i heatmapę korelacji
5. Grupować dane i porównywać kategorie zawodników

## 📋 Wymagania wstępne

- Podstawy Pythona (zmienne, listy, funkcje)
- Podstawy pandas i NumPy (mile widziane, nie wymagane)

---
> 💡 **Wskazówka:** Uruchamiaj każdą komórkę `Shift+Enter`. Jeśli utkniesz, przeczytaj uważnie komunikaty błędów w komórkach testowych — mówią dokładnie, co poszło nie tak!

In [ ]:
# 🔧 Setup — uruchom tę komórkę jako pierwszą!

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 20)
plt.style.use('seaborn-v0_8-whitegrid')

np.random.seed(42)  # 👈 powtarzalność wyników

print('✅ Setup zakończony! Lecimy! 🚀')

## 📖 Zbiór danych: Maraton Amatorski 2026

Wygenerujemy syntetyczny (ale realistyczny!) zbiór danych o ~1300 zawodnikach amatorskiego maratonu.

| Kolumna | Opis | Typ |
|---|---|---|
| zawodnik_id | Unikalny numer startowy | int |
| wiek | Wiek zawodnika (lata) | int |
| plec | Płeć (K/M) | category |
| czas_min | Czas ukończenia maratonu (minuty) | float |
| tempo_min_km | Tempo biegu (minuty/km) | float |
| kategoria_wiekowa | Kategoria wiekowa (18-29, 30-39, ...) | category |
| miejsce | Pozycja w klasyfikacji generalnej | int |
| rok | Rok zawodów | int |

> 🏃 Dystans maratonu to zawsze 42.195 km — to jedna z niewielu stałych w naszym zbiorze danych!

In [ ]:
# 📊 Generujemy dane — syntetyczny maraton amatorski

n = 1300

# Wiek zawodników — rozkład normalny, przycięty do sensownego zakresu
wiek = np.clip(np.random.normal(loc=38, scale=9, size=n), 18, 75).astype(int)

# Płeć — nieco więcej mężczyzn niż kobiet (typowe dla maratonów amatorskich)
plec = np.random.choice(['K', 'M'], size=n, p=[0.42, 0.58])

# Bazowy czas zależy od wieku (odchylenie od 35 lat spowalnia) i płci
bazowy_czas = 230 + (np.abs(wiek - 35) * 0.8) + np.where(plec == 'K', 12, 0)
szum = np.random.normal(loc=0, scale=25, size=n)
czas_min = bazowy_czas + szum

# Kilku zawodników miało bardzo słaby dzień (kontuzja, upał) — realistyczne outliery
outlier_idx = np.random.choice(n, size=15, replace=False)
czas_min[outlier_idx] += np.random.uniform(60, 150, size=15)
czas_min = np.clip(czas_min, 150, None)  # nikt nie biegnie szybciej niż elita

dystans_km = 42.195
tempo_min_km = czas_min / dystans_km

df = pd.DataFrame({
    'zawodnik_id': np.arange(1, n + 1),
    'wiek': wiek,
    'plec': plec,
    'czas_min': czas_min.round(1),
    'tempo_min_km': tempo_min_km.round(2),
})

bins = [17, 29, 39, 49, 59, 100]
labels = ['18-29', '30-39', '40-49', '50-59', '60+']
df['kategoria_wiekowa'] = pd.cut(df['wiek'], bins=bins, labels=labels)

df['miejsce'] = df['czas_min'].rank(method='min').astype(int)
df['rok'] = 2026

print(f'📊 Wygenerowano dane dla {len(df)} zawodników')
df.head()

## 🎓 Przykład prowadzony

Zanim zaczniesz samodzielnie, przejdźmy razem przez pełny przykład: sprawdzimy podstawowe statystyki czasu ukończenia i narysujemy pierwszy histogram.

In [ ]:
# 🎓 Przykład prowadzony — uruchom i przeanalizuj

srednia = df['czas_min'].mean()
mediana = df['czas_min'].median()
odchylenie = df['czas_min'].std()

print(f'Średni czas: {srednia:.1f} min')
print(f'Mediana czasu: {mediana:.1f} min')
print(f'Odchylenie standardowe: {odchylenie:.1f} min')

if abs(srednia - mediana) > 3:
    print('⚠️ Zauważalna różnica średnia-mediana — możliwa skośność lub outliery!')

plt.figure(figsize=(10, 6))
plt.hist(df['czas_min'], bins=30, edgecolor='black', alpha=0.7)
plt.axvline(srednia, color='red', linestyle='--', label=f'Średnia: {srednia:.1f} min')
plt.axvline(mediana, color='green', linestyle='--', label=f'Mediana: {mediana:.1f} min')
plt.xlabel('Czas ukończenia (min)')
plt.ylabel('Liczba zawodników')
plt.title('Rozkład czasów ukończenia maratonu')
plt.legend()
plt.show()

---
## ✏️ Ćwiczenie 1: Pierwszy rzut oka na dane

**Trudność:** ⭐

Sprawdź podstawowe informacje o zbiorze danych `df`.

**Zadanie:**
1. Zapisz liczbę wierszy do zmiennej `liczba_zawodnikow`
2. Zapisz listę nazw kolumn do zmiennej `nazwy_kolumn`
3. Zapisz wynik `df.describe()` do zmiennej `statystyki`

> 💡 **Wskazówka:** `df.shape` zwraca krotkę (wiersze, kolumny). `df.columns` zwraca obiekt Index — zamień go na listę funkcją `list()`.

In [ ]:
# ✏️ Ćwiczenie 1

# TODO: liczba wierszy w df
liczba_zawodnikow = ...

# TODO: lista nazw kolumn
nazwy_kolumn = ...

# TODO: statystyki opisowe całego df
statystyki = ...

print(f'Liczba zawodników: {liczba_zawodnikow}')
print(f'Kolumny: {nazwy_kolumn}')

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 1 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: liczba wierszy w DataFrame to liczba zawodników w zbiorze danych
liczba_zawodnikow = len(df)

# Krok 2: df.columns zwraca obiekt Index — zamieniamy go na zwykłą listę
nazwy_kolumn = list(df.columns)

# Krok 3: describe() liczy statystyki opisowe (mean, std, min, max, kwartyle) dla kolumn numerycznych
statystyki = df.describe()

print(f'Liczba zawodników: {liczba_zawodnikow}')
print(f'Kolumny: {nazwy_kolumn}')

In [ ]:
# ✅ Test — Ćwiczenie 1
try:
    assert liczba_zawodnikow == len(df), '❌ Nieprawidłowa liczba zawodników'
    assert isinstance(nazwy_kolumn, list), '❌ nazwy_kolumn powinna być listą'
    assert 'czas_min' in nazwy_kolumn, '❌ Brakuje kolumny czas_min w liście'
    assert isinstance(statystyki, pd.DataFrame), '❌ statystyki powinny być DataFrame'
    print('✅ Wszystkie testy zaliczone! Świetna robota! 🎉')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! 💪')

---
## ✏️ Ćwiczenie 2: Miary tendencji centralnej

**Trudność:** ⭐

Oblicz średnią, medianę i modę wieku zawodników.

1. `srednia_wieku`
2. `mediana_wieku`
3. `moda_wieku` (pierwsza wartość mody — użyj `.mode()[0]`)

In [ ]:
# ✏️ Ćwiczenie 2

# TODO: średni wiek zawodników
srednia_wieku = ...

# TODO: mediana wieku
mediana_wieku = ...

# TODO: moda wieku (pierwsza wartość)
moda_wieku = ...

print(f'Średnia: {srednia_wieku:.1f}, Mediana: {mediana_wieku:.1f}, Moda: {moda_wieku}')

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 2 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: średnia arytmetyczna wieku zawodników
srednia_wieku = df['wiek'].mean()

# Krok 2: mediana — wartość środkowa po posortowaniu, odporna na outliery
mediana_wieku = df['wiek'].median()

# Krok 3: moda — najczęściej występująca wartość; .mode() może zwrócić kilka wartości, bierzemy pierwszą
moda_wieku = df['wiek'].mode()[0]

print(f'Średnia: {srednia_wieku:.1f}, Mediana: {mediana_wieku:.1f}, Moda: {moda_wieku}')

In [ ]:
# ✅ Test — Ćwiczenie 2
try:
    assert round(srednia_wieku, 1) == round(df['wiek'].mean(), 1), '❌ Nieprawidłowa średnia'
    assert mediana_wieku == df['wiek'].median(), '❌ Nieprawidłowa mediana'
    assert moda_wieku == df['wiek'].mode()[0], '❌ Nieprawidłowa moda'
    print('✅ Wszystkie testy zaliczone! 🎉')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! 💪')

---
## ✏️ Ćwiczenie 3: Miary rozproszenia i IQR

**Trudność:** ⭐⭐

Oblicz wariancję, odchylenie standardowe oraz IQR czasu ukończenia (`czas_min`).

1. `wariancja_czasu`
2. `odchylenie_czasu`
3. `q1`, `q3` (kwantyle 0.25 i 0.75)
4. `iqr` = q3 - q1

In [ ]:
# ✏️ Ćwiczenie 3

# TODO: wariancja czasu ukończenia
wariancja_czasu = ...

# TODO: odchylenie standardowe czasu ukończenia
odchylenie_czasu = ...

# TODO: pierwszy i trzeci kwartyl
q1 = ...
q3 = ...

# TODO: rozstęp międzykwartylowy
iqr = ...

print(f'Std: {odchylenie_czasu:.1f}, IQR: {iqr:.1f}')

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 3 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: wariancja mierzy średni kwadrat odchylenia od średniej
wariancja_czasu = df['czas_min'].var()

# Krok 2: odchylenie standardowe to pierwiastek z wariancji — ta sama jednostka co dane (minuty)
odchylenie_czasu = df['czas_min'].std()

# Krok 3: pierwszy i trzeci kwartyl — granice środkowych 50% obserwacji
q1 = df['czas_min'].quantile(0.25)
q3 = df['czas_min'].quantile(0.75)

# Krok 4: IQR to szerokość "pudełka" na boxplocie — rozstęp międzykwartylowy
iqr = q3 - q1

print(f'Std: {odchylenie_czasu:.1f}, IQR: {iqr:.1f}')

In [ ]:
# ✅ Test — Ćwiczenie 3
try:
    assert round(wariancja_czasu, 1) == round(df['czas_min'].var(), 1), '❌ Nieprawidłowa wariancja'
    assert round(odchylenie_czasu, 1) == round(df['czas_min'].std(), 1), '❌ Nieprawidłowe odchylenie std'
    assert round(q1, 1) == round(df['czas_min'].quantile(0.25), 1), '❌ Nieprawidłowy Q1'
    assert round(q3, 1) == round(df['czas_min'].quantile(0.75), 1), '❌ Nieprawidłowy Q3'
    assert round(iqr, 1) == round(q3 - q1, 1), '❌ IQR powinien być równy Q3 - Q1'
    print('✅ Wszystkie testy zaliczone! 🎉')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! 💪')

---
## ✏️ Ćwiczenie 4: Wykrywanie outlierów metodą IQR

**Trudność:** ⭐⭐

Korzystając z `q1`, `q3` i `iqr` z poprzedniego ćwiczenia, znajdź zawodników z nietypowo długim czasem ukończenia.

1. `granica_gorna` = q3 + 1.5*iqr
2. `outliers` — DataFrame z wierszami, gdzie `czas_min > granica_gorna`
3. `liczba_outlierow` — liczba takich zawodników

In [ ]:
# ✏️ Ćwiczenie 4

# TODO: górna granica outlierów (metoda IQR)
granica_gorna = ...

# TODO: wiersze df, gdzie czas_min przekracza górną granicę
outliers = ...

# TODO: liczba outlierów
liczba_outlierow = ...

print(f'Górna granica: {granica_gorna:.1f} min, liczba outlierów: {liczba_outlierow}')

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 4 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: górna granica wg klasycznej reguły Tukeya — 1.5 * IQR powyżej trzeciego kwartyla
granica_gorna = q3 + 1.5 * iqr

# Krok 2: wybieramy wiersze, których czas ukończenia przekracza tę granicę
outliers = df[df['czas_min'] > granica_gorna]

# Krok 3: liczba takich zawodników to po prostu liczba wierszy w outliers
liczba_outlierow = len(outliers)

print(f'Górna granica: {granica_gorna:.1f} min, liczba outlierów: {liczba_outlierow}')

In [ ]:
# ✅ Test — Ćwiczenie 4
try:
    oczekiwana_granica = q3 + 1.5 * iqr
    assert round(granica_gorna, 1) == round(oczekiwana_granica, 1), '❌ Nieprawidłowa górna granica'
    assert isinstance(outliers, pd.DataFrame), '❌ outliers powinien być DataFrame'
    assert liczba_outlierow == len(outliers), '❌ liczba_outlierow nie zgadza się z outliers'
    assert (outliers['czas_min'] > granica_gorna).all(), '❌ outliers zawiera nieprawidłowe wiersze'
    print('✅ Wszystkie testy zaliczone! Znalazłeś prawdziwe outliery! 🎉')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! 💪')

---
## ✏️ Ćwiczenie 5: Histogram tempa biegu

**Trudność:** ⭐

Narysuj histogram kolumny `tempo_min_km` i zaznacz linią średnią.

Wymagania: `bins=30`, tytuł, etykiety osi, `plt.axvline` ze średnią, `plt.legend()`.

In [ ]:
# ✏️ Ćwiczenie 5

# TODO: oblicz średnie tempo
srednie_tempo = ...

plt.figure(figsize=(10, 6))
# TODO: narysuj histogram tempa (kolumna 'tempo_min_km', bins=30)
#       dodaj plt.axvline ze średnią, plt.xlabel, plt.ylabel, plt.title, plt.legend

plt.show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 5 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: średnie tempo potrzebne do narysowania pionowej linii na histogramie
srednie_tempo = df['tempo_min_km'].mean()

plt.figure(figsize=(10, 6))
# Krok 2: histogram rozkładu tempa — 30 przedziałów (bins) dla czytelności
plt.hist(df['tempo_min_km'], bins=30, edgecolor='black', alpha=0.7)
# Krok 3: pionowa przerywana linia zaznaczająca średnie tempo
plt.axvline(srednie_tempo, color='red', linestyle='--', label=f'Średnia: {srednie_tempo:.2f} min/km')
# Krok 4: opisy osi, tytuł i legenda — bez nich wykres jest nieczytelny
plt.xlabel('Tempo (min/km)')
plt.ylabel('Liczba zawodników')
plt.title('Rozkład tempa biegu')
plt.legend()

plt.show()

In [ ]:
# ✅ Test — Ćwiczenie 5
try:
    assert round(srednie_tempo, 2) == round(df['tempo_min_km'].mean(), 2), '❌ Nieprawidłowe średnie tempo'
    print('✅ Test zaliczony! Sprawdź wizualnie, czy Twój wykres ma tytuł, etykiety i legendę 🎨')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! 💪')

---
## ✏️ Ćwiczenie 6: Boxplot — czas wg kategorii wiekowej

**Trudność:** ⭐⭐

Użyj `seaborn.boxplot`, aby porównać czasy ukończenia (`czas_min`) w różnych kategoriach wiekowych (`kategoria_wiekowa`).

Zapisz medianę czasu dla każdej kategorii do zmiennej `mediany_kategorii` (użyj `groupby` + `median`).

In [ ]:
# ✏️ Ćwiczenie 6

# TODO: mediana czasu ukończenia w każdej kategorii wiekowej
mediany_kategorii = ...

plt.figure(figsize=(10, 6))
# TODO: narysuj boxplot (x='kategoria_wiekowa', y='czas_min', data=df)

plt.title('Czas ukończenia wg kategorii wiekowej')
plt.xlabel('Kategoria wiekowa')
plt.ylabel('Czas ukończenia (min)')
plt.show()

print(mediany_kategorii)

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 6 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: mediana czasu w każdej kategorii wiekowej (observed=True pomija puste kategorie)
mediany_kategorii = df.groupby('kategoria_wiekowa', observed=True)['czas_min'].median()

plt.figure(figsize=(10, 6))
# Krok 2: boxplot pokazuje medianę, kwartyle i outliery dla każdej grupy jednocześnie
sns.boxplot(x='kategoria_wiekowa', y='czas_min', data=df)

plt.title('Czas ukończenia wg kategorii wiekowej')
plt.xlabel('Kategoria wiekowa')
plt.ylabel('Czas ukończenia (min)')
plt.show()

print(mediany_kategorii)

In [ ]:
# ✅ Test — Ćwiczenie 6
try:
    oczekiwane = df.groupby('kategoria_wiekowa', observed=True)['czas_min'].median()
    assert mediany_kategorii.round(1).equals(oczekiwane.round(1)), '❌ Nieprawidłowe mediany kategorii'
    print('✅ Test zaliczony! 🎉')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! 💪')

---
## ✏️ Ćwiczenie 7: Korelacja i heatmapa

**Trudność:** ⭐⭐

Oblicz macierz korelacji dla kolumn `['wiek', 'czas_min', 'tempo_min_km']` i narysuj heatmapę.

Zapisz macierz do `macierz_korelacji`.

In [ ]:
# ✏️ Ćwiczenie 7

kolumny_numeryczne = ['wiek', 'czas_min', 'tempo_min_km']

# TODO: macierz korelacji dla kolumny_numeryczne
macierz_korelacji = ...

plt.figure(figsize=(8, 6))
sns.heatmap(macierz_korelacji, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Macierz korelacji')
plt.show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 7 (kliknij, aby zobaczyć krok po kroku)

kolumny_numeryczne = ['wiek', 'czas_min', 'tempo_min_km']

# Krok 1: macierz korelacji Pearsona — jak silnie zmienne numeryczne są ze sobą powiązane
macierz_korelacji = df[kolumny_numeryczne].corr()

plt.figure(figsize=(8, 6))
# Krok 2: heatmapa z adnotacjami liczbowymi i symetryczną skalą kolorów wokół 0
sns.heatmap(macierz_korelacji, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Macierz korelacji')
plt.show()

In [ ]:
# ✅ Test — Ćwiczenie 7
try:
    oczekiwana = df[['wiek', 'czas_min', 'tempo_min_km']].corr()
    assert macierz_korelacji.round(2).equals(oczekiwana.round(2)), '❌ Nieprawidłowa macierz korelacji'
    print('✅ Test zaliczony! 🎉')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! 💪')

---
## ✏️ Ćwiczenie 8: Grupowanie i agregacja

**Trudność:** ⭐⭐⭐

Sprawdź, czy istnieje różnica w średnim czasie ukończenia między kobietami a mężczyznami.

1. `agregacja_plec` — DataFrame z `groupby('plec')` agregujący `czas_min` funkcjami `['mean', 'std', 'count']`
2. `roznica_plec` — wartość bezwzględna różnicy średnich czasów K vs M

In [ ]:
# ✏️ Ćwiczenie 8

# TODO: agregacja czasu ukończenia wg płci (mean, std, count)
agregacja_plec = ...

# TODO: różnica bezwzględna średnich czasów między płciami
roznica_plec = ...

print(agregacja_plec)
print(f'Różnica średnich czasów K vs M: {roznica_plec:.1f} min')

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 8 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: agregacja czasu ukończenia wg płci trzema funkcjami naraz (mean, std, count)
agregacja_plec = df.groupby('plec')['czas_min'].agg(['mean', 'std', 'count'])

# Krok 2: różnica bezwzględna średnich czasów między kobietami (K) a mężczyznami (M)
roznica_plec = abs(agregacja_plec.loc['K', 'mean'] - agregacja_plec.loc['M', 'mean'])

print(agregacja_plec)
print(f'Różnica średnich czasów K vs M: {roznica_plec:.1f} min')

In [ ]:
# ✅ Test — Ćwiczenie 8
try:
    oczekiwane = df.groupby('plec')['czas_min'].agg(['mean', 'std', 'count'])
    assert agregacja_plec.round(1).equals(oczekiwane.round(1)), '❌ Nieprawidłowa agregacja'
    oczekiwana_roznica = abs(oczekiwane.loc['K', 'mean'] - oczekiwane.loc['M', 'mean'])
    assert round(roznica_plec, 1) == round(oczekiwana_roznica, 1), '❌ Nieprawidłowa różnica'
    print('✅ Wszystkie testy zaliczone! Widzisz jakąś różnicę? Sprawdź ją statystycznie w zadaniu bonusowym! 🎉')
except AssertionError as e:
    print(str(e))
    print('Spróbuj ponownie! 💪')

---
## 🏆 Wyzwanie bonusowe (opcjonalne)

Zbadaj rozkład tempa biegu (`tempo_min_km`) osobno dla kategorii `18-29` i `60+` i sprawdź, czy najstarsza kategoria biegnie istotnie wolniej. Wykorzystaj `scipy.stats.ttest_ind` (intuicję poznaliśmy na końcu Wykładu 1) i policz podstawowe statystyki grup.

In [ ]:
# 🏆 Bonus — Twój kod tutaj

grupa_mloda = df[df['kategoria_wiekowa'] == '18-29']['tempo_min_km']
grupa_starsza = df[df['kategoria_wiekowa'] == '60+']['tempo_min_km']

print(f'Średnie tempo 18-29: {grupa_mloda.mean():.2f} min/km')
print(f'Średnie tempo 60+: {grupa_starsza.mean():.2f} min/km')

statystyka, p_wartosc = stats.ttest_ind(grupa_mloda, grupa_starsza, equal_var=False)
print(f'p-wartość testu t: {p_wartosc:.4f}')

if p_wartosc < 0.05:
    print('🔍 Różnica wygląda na statystycznie istotną — p < 0.05, odrzucamy H0!')
else:
    print('🔍 Różnica może być dziełem przypadku — p ≥ 0.05, brak podstaw do odrzucenia H0.')

---
## 📝 Podsumowanie

Świetna robota — oto, co przećwiczyłeś:

- ✅ Wczytywanie i eksploracja danych (`shape`, `info`, `describe`)
- ✅ Miary tendencji centralnej (średnia, mediana, moda)
- ✅ Miary rozproszenia (wariancja, odchylenie std, kwantyle, IQR)
- ✅ Wykrywanie outlierów metodą IQR
- ✅ Wizualizacja: histogram, boxplot, heatmapa korelacji
- ✅ Grupowanie i agregacja danych (`groupby`)

### 📚 Dalsza lektura
- [Dokumentacja pandas — statystyki opisowe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
- *Practical Statistics for Data Scientists* — Peter Bruce & Andrew Bruce

### 🔮 Co dalej?
Na Wykładzie 2 (13.10.2026) wchodzimy w świat **Big Data** — zobaczysz, jak przechowywać i czytać dane, gdy zamiast 1 300 zawodników masz miliardy rekordów: CSV vs Parquet, kompresja i partycjonowanie! 📦

Do zobaczenia na kolejnych zajęciach! 🎉